# Lesson 4.3 - CPU Serving, HPA, And OpenWebUI

**Level: Intermediate**

## Learning Objectives
- Deploy CPU Ollama workloads (TinyLlama and Qwen variants).
- Configure Horizontal Pod Autoscaler for TinyLlama service.
- Publish OpenWebUI as public entrypoint while keeping model backends internal.

**Project file(s) produced:** `gitops/ollama-tinyllama/*`, `gitops/ollama-qwen/*`, `gitops/openwebui/*`

In [ ]:
%%writefile gitops/ollama-tinyllama/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: ollama
  namespace: ai-demos
spec:
  type: ClusterIP
  selector:
    app: ollama
  ports:
    - name: http
      port: 11434
      targetPort: 11434

In [ ]:
%%writefile gitops/ollama-tinyllama/hpa.yaml
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ollama
  namespace: ai-demos
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ollama
  minReplicas: 1
  maxReplicas: 3
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 50

In [ ]:
%%writefile gitops/openwebui/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: openwebui
  namespace: ai-demos
spec:
  replicas: 1
  selector:
    matchLabels:
      app: openwebui
  template:
    metadata:
      labels:
        app: openwebui
    spec:
      containers:
        - name: openwebui
          image: ghcr.io/open-webui/open-webui:main
          env:
            - name: OLLAMA_BASE_URL
              value: "http://ollama:11434"
            - name: OPENAI_API_BASE_URL
              value: "http://vllm-serv:8000/v1"
            - name: OPENAI_API_KEY
              value: "not-required"

In [ ]:
%%writefile gitops/openwebui/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: openwebui
  namespace: ai-demos
spec:
  type: LoadBalancer
  selector:
    app: openwebui
  ports:
    - name: http
      port: 80
      targetPort: 8080

## Exercise
Complete remaining workload manifests to match repo behavior: tinyllama deployment, qwen deployment/service, and all workload kustomization files.

In [ ]:
# TODO: Create gitops/ollama-tinyllama/deployment.yaml
# TODO: Create gitops/ollama-qwen/deployment.yaml and service.yaml
# TODO: Create kustomization.yaml in each workload directory
print('Then verify ArgoCD sync status and test both CPU and GPU-backed models in OpenWebUI.')

---
**Next:** [Capstone Project](../capstone/capstone-project.ipynb)